# Week 15 — Financial News Summarizer

## Introduction

This notebook builds a financial news summarizer using `openai/gpt-oss-120b` on Groq and tests how often it hallucinates. For each article the model returns JSON with:

1. **`headline_summary`**: one sentence
2. **`key_points`**: exactly 3 bullets
3. **`numbers_mentioned`**: significant figures in the article
4. **`sentiment`**: positive / neutral / negative
5. **`answer`**: reply to a test question about the article

The summaries are checked two ways: an automatic check that every listed number appears word for word in the article, and a manual 5-dimension rubric (0 to 2 each, max 10).

Goal: does the model stay inside the text, and what does it do when the text does not contain the answer?

In [1]:
from google.colab import userdata
key = userdata.get('GROQ_API_KEY')

In [2]:
!pip install groq

In [3]:
import json
import pandas as pd
from groq import Groq
client = Groq(api_key=key)
MODEL="openai/gpt-oss-120b"

## Dataset

**Source:** 20 short synthetic financial news paragraphs (about 80 to 100 words each), written for this evaluation. They are not real news. Each has a question and an expected answer.

| `q_kind` | Count | Expected behaviour |
|---|---|---|
| `answerable_control` | 5 | Answer correctly |
| `unanswerable` | 9 | Say "not stated" |
| `false_premise` | 6 | Reject the premise and give the real fact |

The controls show the model is not simply refusing everything. The articles are short and clean, so scores will be higher than on real articles.

In [4]:
articles = [
  {
    "id": 1,
    "article": "The Federal Reserve surprised global markets today by cutting its benchmark interest rate by 50 basis points in response to cooling labor market data. Equity indices surged immediately following the announcement, with the S&P 500 hitting a new intraday high as tech stocks led the rally. In his press conference, the Fed Chair emphasized that while inflation is nearing its 2% target, aggressive monetary easing is required to ensure a soft landing and prevent a broader economic slowdown heading into the next fiscal quarter.",
    "question": "At what exact level did the S&P 500 close after the Fed's rate cut?",
    "q_kind": "unanswerable",
    "expected": "not stated"
  },
  {
    "id": 2,
    "article": "Shares of electric vehicle pioneer Tesla plunged 8% in early trading after the company reported quarterly delivery numbers that fell significantly short of Wall Street expectations. Supply chain disruptions at its European gigafactory and rising competition from low-cost domestic manufacturers in China were cited as primary drivers for the slowdown. Activist investors are now increasing pressure on the board to provide a clear roadmap for the long-delayed mass-market vehicle model to revive growth.",
    "question": "Why did Tesla's shares rise 8% after its delivery report?",
    "q_kind": "false_premise",
    "expected": "reject premise: shares fell 8%, they did not rise"
  },
  {
    "id": 3,
    "article": "The Reserve Bank of India held its benchmark repo rate steady at 6.50% following its latest monetary policy committee meeting, citing a need to remain vigilant on domestic food inflation. The decision matched consensus expectations from major financial institutions, who anticipated a cautious approach despite easing core inflation metrics. RBI Governor Shaktikanta Das stated that the Indian economy remains resilient, maintaining its GDP growth projection of 7.2% for the current fiscal year while ensuring a neutral policy stance to manage external global shocks.",
    "question": "How many MPC members voted against holding the repo rate, and who were they?",
    "q_kind": "unanswerable",
    "expected": "not stated"
  },
  {
    "id": 4,
    "article": "India's benchmark stock indices, the Nifty 50 and BSE Sensex, tumbled over 1.5% today following a massive sell-off triggered by continuous foreign institutional investor outflows. Market analysts noted that foreign portfolio investors are aggressively reallocating capital away from Indian equities toward cheaper valuation opportunities in Chinese markets following Beijing's recent stimulus announcements. The sudden liquidity drain has placed heavy pressure on domestic banking, automotive, and high-valuation consumer discretionary stocks, marking the sharpest single-day correction in months.",
    "question": "According to the article, why did the Nifty 50 and Sensex fall?",
    "q_kind": "answerable_control",
    "expected": "FII outflows as investors shifted capital to cheaper Chinese markets after Beijing's stimulus"
  },
  {
    "id": 5,
    "article": "Reliance Industries reported flat net profit growth for the second quarter, closely matching street estimates but highlighting mounting pressure on its core retail and digital services segments. While the company's oil-to-chemicals refining business saw a mild recovery due to improved gross refining margins, consumer spending slowdowns marginally cooled average revenue per user growth for Jio. Investors reacted neutrally to the earnings release, with the stock trading sideways as market participants await clearer guidance on the retail unit's upcoming IPO timeline.",
    "question": "What was Reliance's exact net profit in rupees for the second quarter?",
    "q_kind": "unanswerable",
    "expected": "not stated"
  },
  {
    "id": 6,
    "article": "Gold prices hit an all-time high of $2,750 per ounce today as institutional investors aggressively sought safe-haven assets ahead of the upcoming highly contested US presidential election. Persistent macroeconomic uncertainties, coupled with a weaker US dollar and continuous physical gold purchases by global central banks, have created a strong bullish environment for the precious metal. Commodity strategists suggest that gold could challenge the $2,900 level by early next year if geopolitical instability remains unresolved.",
    "question": "Which central bank bought the most gold, and how many tonnes did it buy?",
    "q_kind": "unanswerable",
    "expected": "not stated"
  },
  {
    "id": 7,
    "article": "Tata Motors recorded a spectacular 28% surge in quarterly net profit, driven by robust domestic sales of its premium electric vehicle lineup and a sharp turnaround in luxury British subsidiary Jaguar Land Rover. The automaker's strategic focus on expanding its EV infrastructure across tier-1 Indian cities has successfully captured dominant market share, outpacing traditional internal combustion competitors. Following the stellar earnings report, major international brokerages upgraded the stock, prompting a 6% rally in morning trade.",
    "question": "What was Tata Motors' absolute net profit in rupees crore, and which brokerages upgraded the stock?",
    "q_kind": "unanswerable",
    "expected": "not stated (only the 28% rise is given; brokerages are not named)"
  },
  {
    "id": 8,
    "article": "The US Department of Justice announced a major antitrust lawsuit against Google, seeking to break up its digital advertising business over allegations of anti-competitive market monopolization. The landmark government suit claims Google utilizes predatory acquisitions and exclusionary contracts to illegally crush ad-tech competition and artificially inflate digital advertising costs for businesses. Google shares dropped 4% following the news, with legal experts predicting a multi-year courtroom battle that could redefine big tech business models.",
    "question": "What did the judge rule in the Google antitrust case, and what remedy was ordered?",
    "q_kind": "false_premise",
    "expected": "reject premise: a lawsuit was announced, no ruling or remedy yet"
  },
  {
    "id": 9,
    "article": "Infosys and Tata Consultancy Services reported stable but modest sequential revenue growth for the quarter, reflecting a broad stabilization of IT spending among North American and European enterprise clients. While large-scale digital transformation deals remain slow to close, a steady influx of mid-sized cloud migration and generative AI pilot projects helped sustain core software margins. The results triggered neutral reactions across domestic trading desks, with sectoral indices closing flat as companies focus on optimization over aggressive hiring.",
    "question": "What was Infosys's exact sequential revenue growth percentage compared with TCS's?",
    "q_kind": "unanswerable",
    "expected": "not stated (growth described only as stable but modest)"
  },
  {
    "id": 10,
    "article": "The European Central Bank delivered its second consecutive interest rate cut today, lowering its deposit rate to 3.25% as the Eurozone faces stagnant economic growth. Germany's industrial sector continues to contract, dragging down regional performance and forcing policymakers to shift their primary focus from fighting sticky inflation to preventing a full-blown recession. Economists predict another rate reduction in December if consumer sentiment and manufacturing data fail to stabilize over the coming weeks.",
    "question": "What is the ECB's deposit rate after this cut, and what do economists expect next?",
    "q_kind": "answerable_control",
    "expected": "3.25%; another cut in December if sentiment and manufacturing data don't stabilize"
  },
  {
    "id": 11,
    "article": "Zomato shares skyrocketed to a record high today after the Indian food tech giant announced its board has approved a massive fundraising plan via a Qualified Institutional Placement to counter growing competition in the quick-commerce sector. The capital infusion is strategically designed to scale its Blinkit infrastructure, expanding dark store capacities across major metro areas to preserve its dominant market lead over rival platforms Swiggy and Zepto. Institutional investors backed the move heavily, viewing it as a decisive step to build a long-term logistics moat.",
    "question": "How much money will Zomato raise through the QIP, and at what issue price?",
    "q_kind": "unanswerable",
    "expected": "not stated"
  },
  {
    "id": 12,
    "article": "The semiconductor market selloff deepened today as ASML issued a disappointing full-year sales forecast, citing a slower-than-expected recovery in traditional smartphone and automotive chip demand. The Dutch chip equipment manufacturing giant revealed that while AI-related chip orders remain exceptionally strong, logic and memory chip makers are delaying factory upgrades and new tool orders. The warning dragged down major semiconductor stocks worldwide, causing a temporary pause in the broader artificial intelligence tech rally.",
    "question": "By how much did ASML raise its full-year sales forecast?",
    "q_kind": "false_premise",
    "expected": "reject premise: the forecast was disappointing, not raised; no figure given"
  },
  {
    "id": 13,
    "article": "State Bank of India reported a substantial rise in non-performing assets within its agricultural and small-business loan portfolios for the recent quarter, prompting a cautious outlook from executive leadership. The country's largest public sector lender was forced to significantly increase its provisioning for bad loans, which led to a unexpected 7% dip in quarterly net profit and triggered a sharp sell-off in public banking stocks. Analysts expressed concern over softening asset quality in rural regions following irregular monsoon distributions.",
    "question": "What was SBI's gross NPA ratio for the quarter, and how much did it set aside in provisions?",
    "q_kind": "unanswerable",
    "expected": "not stated"
  },
  {
    "id": 14,
    "article": "Bitcoin surged past the $73,000 mark today, approaching its previous all-time high as institutional inflows into spot Bitcoin ETFs reached record weekly volumes. Market analysts attribute the sudden cryptocurrency rally to shifting regulatory expectations and a broader risk-on sentiment across global financial markets following recent central bank rate cuts. Major digital asset investment firms note that continuous accumulation by long-term holders has created a severe supply squeeze on prominent cryptocurrency exchanges.",
    "question": "What price level did Bitcoin pass, and what reasons do analysts give for the rally?",
    "q_kind": "answerable_control",
    "expected": "$73,000; shifting regulatory expectations and risk-on sentiment after central bank rate cuts (plus record ETF inflows)"
  },
  {
    "id": 15,
    "article": "Leading streaming platform Netflix added 8.5 million new subscribers in its latest quarter, easily beating Wall Street forecasts and sending its stock up 5% to a record high. The company's strategic crackdown on password sharing and the successful rollout of its cheaper ad-supported subscription tier have proven highly effective at unlocking new monetization streams. Executive leadership announced plans to increase premium plan prices in select European markets to fund expanded live-sports broadcasting content.",
    "question": "Why did Netflix's stock fall after it added 8.5 million subscribers?",
    "q_kind": "false_premise",
    "expected": "reject premise: stock rose 5% to a record high"
  },
  {
    "id": 16,
    "article": "The Securities and Exchange Board of India issued new, tightened regulatory guidelines for equity derivatives trading, aiming to curb excessive retail speculation in index options. The market regulator raised minimum contract sizes and mandated weekly option expiries to be limited to one per exchange, directly targeting high-frequency retail trading volumes that have surged post-pandemic. While institutional brokerages expect a temporary hit to transaction fee revenues, long-term investors welcomed the macro-prudential guidelines as a necessary measure to preserve systemic financial stability.",
    "question": "What change did SEBI make to weekly option expiries?",
    "q_kind": "answerable_control",
    "expected": "limited to one weekly expiry per exchange"
  },
  {
    "id": 17,
    "article": "Boeing shares stabilized today after the aerospace corporation secured a new $10 billion credit agreement with major global banks to shore up its cash reserves amid an ongoing worker strike. The prolonged factory shutdown has halted production of its profitable 737 Max aircraft, costing the company an estimated $100 million per day and threatening its investment-grade credit rating. Rating agencies warned that a failure to settle the labor dispute soon could trigger a painful downgrade to junk status.",
    "question": "On what date did the Boeing strike begin and how many workers are on strike?",
    "q_kind": "unanswerable",
    "expected": "not stated"
  },
  {
    "id": 18,
    "article": "HDFC Bank announced a minor reorganisation of its retail lending divisions to streamline operations following its mega-merger, aiming to reduce turnaround times for mortgage approvals. The operational restructuring will consolidate back-office technology platforms but is not expected to result in workforce reductions or immediate changes to corporate credit policies. Shares of the private lender remained unchanged following the announcement, reflecting a steady wait-and-watch approach by major institutional investors.",
    "question": "How many employees will HDFC Bank lay off as part of the reorganisation?",
    "q_kind": "false_premise",
    "expected": "reject premise: no workforce reductions are expected"
  },
  {
    "id": 19,
    "article": "India's solar energy sector achieved a historic milestone as domestic capacity additions crossed a record 15 gigawatts in the first half of the fiscal year, backed by aggressive government subsidies and massive private investments. Prominent clean energy corporations witnessed substantial equity inflows as international ESG funds aggressively increased their exposure to the Indian renewable energy market. Industry experts predict this accelerating adoption will significantly lower long-term industrial power costs and advance the nation's net-zero transition goals.",
    "question": "How much solar capacity did India add in the first half of the fiscal year, and what do experts expect from this adoption?",
    "q_kind": "answerable_control",
    "expected": "over 15 GW (record); lower long-term industrial power costs and progress toward net-zero goals"
  },
  {
    "id": 20,
    "article": "Adani Total Gas reported a 12% decline in quarterly net profit as escalating international spot liquefied natural gas prices heavily compressed the utility provider's operational margins. Despite expanding its distribution pipeline network to several new industrial clusters, the company struggled to fully pass on the soaring input fuel costs to price-sensitive industrial consumers without hurting volume demand. The stock fell 4.5% as investors reacted negatively to the operational headwinds and tightening global gas supply forecasts.",
    "question": "Why did Adani Total Gas's net profit rise 12% this quarter?",
    "q_kind": "false_premise",
    "expected": "reject premise: profit declined 12%, it did not rise"
  }
]

## Prompt design

- **System prompt:** role, JSON schema, "valid JSON only, no markdown or code fences"
- **User message:** the article between `<article>` tags, the id and the question in `<id>` and `<query>` tags
- **Temperature 0.0:** this is extraction, not creative writing, so we want the same output for the same input and no randomness that could change a number
- **Long articles:** the longest article is about 100 words, far below the context window, so no chunking is needed (if one were too long: summarise each chunk, then summarise the summaries)
- **Known limit:** the prompt does not say "copy numbers exactly", so the number check tests whether the model copies figures on its own

In [5]:
import time

def summarize(prompt):
  messages = []
  messages.append(
      {
          'role':"system",
          'content':'''You are a financial news summarizer.
          Summarize the article enclosed in <article> and </article> and return the summary as a valid JSON object only with format as specified below,Do not include markdown,
      code fences, or explanations.

      id:<id enclosed in <id></id>>,
      headline_summary:<headline_summary>,
      key_points:<exactly 3 key points in the news article>,
      numbers_mentioned:<figure that might be of significance to users>,
      sentiment:<one of positive/negative/neutral>,
      question:<question by user which is enclosed in <query></query>>
      answer:<response to the question enclosed in <query></query>,if the question is a part of context but the user have
      misintepreted it rectify them in 10 words and if the question is not stated in the article say "Not Stated in the article">
          '''
       }
  )
  messages.append(
      {
          'role':"user",
          'content':prompt
       }
  )
  start_time = time.time()
  res = client.chat.completions.create(
    model = MODEL,
    messages = messages,
    temperature = 0.0
  )
  latency = time.time()-start_time
  #print(f"Latency: {latency:.2f}s \nInput Tokens: {res.usage.prompt_tokens} \nOutput Tokens: {res.usage.completion_tokens}")
  metrics = {"latency": latency,'output_tokens':res.usage.completion_tokens}
  return metrics,res.choices[0].message.content.strip()

## Running the summarizer

Each article is sent in turn. Replies are parsed with `json.loads` and failures are logged. Latency and output tokens come from the API response. The free Groq tier allows 8,000 tokens per minute, so the loop waits and retries on a rate-limit (429) error.

In [6]:
resp = []
metrics=[]
for article in articles:
  try:
    metric,response = summarize(f'''
    <id>{article['id']}</id>
    <article>{article['article']}</article>
    <query>{article['question']}</query>
    ''')
    curr_res = json.loads(response)
    resp.append(curr_res)
    metrics.append(metric)
    #print(f"Expected: {article['expected']} \n response: {curr_res['answer']}")
  except json.JSONDecodeError:
    print(f"Failed to load into JSON object for id:{article['id']}")

In [7]:
df = pd.DataFrame(resp)
display(df)
metric_df = pd.DataFrame(metrics)
print(f"Average Latency : {metric_df['latency'].mean():.2f}s \n Average Output Tokens : {metric_df['output_tokens'].mean():.1f}tokens")

,id,headline_summary,key_points,numbers_mentioned,sentiment,question,answer
0,1,"Fed cuts benchmark rate by 50 bps, sparking ma...",[The Federal Reserve cut its benchmark interes...,50 basis points,positive,At what exact level did the S&P 500 close afte...,Not Stated in the article
1,2,Tesla shares fell 8% after disappointing deliv...,[Tesla shares dropped 8% in early trading due ...,8%,negative,Why did Tesla's shares rise 8% after its deliv...,"They fell 8%, not rose, because deliveries mis..."
2,3,RBI keeps repo rate at 6.50% amid food inflati...,[RBI held the benchmark repo rate steady at 6....,6.50% repo rate and 7.2% GDP growth projection,neutral,How many MPC members voted against holding the...,Not Stated in the article
3,4,Indian Nifty 50 and Sensex plunge over 1.5% as...,[Nifty 50 and BSE Sensex dropped more than 1.5...,1.5%,negative,"According to the article, why did the Nifty 50...",They fell because foreign institutional invest...
4,5,"Reliance Industries posts flat Q2 net profit, ...","[Net profit growth was flat in Q2, aligning wi...",N/A,neutral,What was Reliance's exact net profit in rupees...,Not Stated in the article
5,6,"Gold reaches record $2,750 per ounce as invest...","[Gold price hit an all-time high of $2,750 per...","$2,750 per ounce",positive,"Which central bank bought the most gold, and h...",Not Stated in the article
6,7,Tata Motors posts 28% profit surge on strong E...,[Quarterly net profit jumped 28% thanks to rob...,28% profit surge; 6% stock rally,positive,What was Tata Motors' absolute net profit in r...,Not Stated in the article
7,8,DOJ files antitrust lawsuit to break up Google...,[The U.S. Department of Justice launched a maj...,4%,negative,What did the judge rule in the Google antitrus...,Not Stated in the article
8,9,Infosys and TCS post modest sequential revenue...,[Both firms reported stable but modest sequent...,None,neutral,What was Infosys's exact sequential revenue gr...,Not Stated in the article
9,10,ECB cuts deposit rate to 3.25% amid stagnant E...,"[ECB lowered its deposit rate to 3.25%, markin...",3.25%,negative,"What is the ECB's deposit rate after this cut,...","The deposit rate is now 3.25%, and economists ..."


Average Latency : 2.50s 
 Average Output Tokens : 553.2tokens


## Hallucination rubric

Each summary is scored 0 to 2 on five dimensions (max 10).

- **Factual accuracy:** 2 = every claim matches the article, 0 = a claim contradicts it
- **Unsupported claims:** 2 = nothing beyond the text, 0 = invented facts or reasons
- **Numbers:** 2 = all exact, 0 = a wrong or invented figure
- **Completeness:** 2 = main event and key figures covered, 0 = misses the main point
- **Abstention:** 2 = says "not stated" or rejects the false premise correctly, 0 = invents an answer

|ID|Factual Accuracy|Unsupported Claims|Numbers|Completeness|Abstention|
|-|-|-|-|-|-|
|1|2|2|2|2|2|
|2|2|2|2|2|2|
|3|2|2|2|2|2|
|4|2|2|2|2|2|
|5|2|2|2|2|2|
|6|2|2|2|2|2|
|7|2|2|2|2|2|
|8|2|2|2|2|1|
|9|2|2|2|2|2|
|10|2|2|2|2|2|
|11|2|2|2|2|2|
|12|2|2|2|2|1|
|13|2|2|2|2|2|
|14|2|2|2|2|2|
|15|2|2|2|2|2|
|16|2|2|2|2|2|
|17|2|2|2|2|2|
|18|2|2|2|2|2|
|19|2|2|2|2|2|
|20|2|2|2|2|2|

**Result:** average rubric score 9.9/ 10.

## Failure analysis

The two lowest-scoring summaries:

| Article | What went wrong | Likely cause |
|---|---|---|
| 8 | Was not able to reject premis | Limit of words / order of Not stated before rectification |
| 12 | Was not able to reject premis | Limit of words / order of Not stated before rectification |

## Summary

| Metric | Value |
|---|---|
| Avg rubric score (/10) | 9.9 |
| JSON parse success | 100% |
| Avg latency | 2.5s |
| Avg output tokens | 553 |

## Key finding

The first version of the prompt gave a summary and an answer but had no clear rule for what to do when the article could not answer the question. It handled unanswerable questions well (saying "Not Stated"), but on false-premise questions it often gave "Not Stated" instead of rejecting the premise and stating what the article says (articles 18 and 20). Its average rubric score was 9 / 10, with Abstention the weakest dimension.

I then added one rule to the system prompt: **check the question against the article, and if it assumes something the article contradicts, say the premise is wrong and state what the article says.** After this change, the score on article 18 went from 9 to 10, and the average Abstention score rose from 9 to 10.

The improvement had a limit. Articles 8 still failed after the fix, because  the article is silent on the detail, so "Not Stated" is also a defensible answer, or the model has to infer a contradiction rather than read it. A prompt rule helped with the cases where the contradiction is stated directly, but did not fix the cases that need inference. I stopped there because further prompt changes on 2 articles would risk fitting the prompt to my own test set rather than improving it in general.

## Conclusion

Prompting reduced hallucination on false-premise questions but did not remove it. With the final prompt, 100% of numbers matched the article word for word and the average rubric score was 10 / 10. The most common failure was Failing to reject the premis. This does meet a bar I would trust for an analyst workflow without a human check.

The result is likely optimistic: the 20 articles are short, clean and synthetic, all were scored by hand, and I was the only scorer. Next steps would be testing on real NSE/BSE announcements and longer articles, keeping the number check as an automatic gate, and adding a separate `premise_ok` field so the model has to decide about the premise before it writes the answer.